In [ ]:
import pandas as pd
import os
import bed_reader
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report, 
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from scipy.stats import mannwhitneyu, chi2_contingency
from datetime import datetime
import json
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

import glob
import re

### let's read in the SBP and DBP snps for people on drugs and run simple association tests for each of the SNPs with each of the phenos:

In [ ]:
SBP_snps_all = pd.read_csv(
    'twoyears_on_drugs_SBP/cusk/e3_l3_d1/merged_blocks_selected_markers.tsv',
    sep = '\t'
)

In [ ]:
SBP_snps_all = SBP_snps_all[SBP_snps_all['phenotype'] == 'SBP_pre_ADJ']

In [ ]:
DBP_snps_all = pd.read_csv(
    'twoyears_on_drugs_DBP/cusk/e3_l3_d1/merged_blocks_selected_markers.tsv',
    sep = '\t'
)

In [ ]:
DBP_snps_all = DBP_snps_all[DBP_snps_all['phenotype'] == 'DBP_pre_ADJ']

In [ ]:
SBP_snps_5 = pd.read_csv(
    'prepost_sbp/cusk/e3_l3_d1/merged_blocks_selected_markers.tsv',
    sep = '\t'
)

In [ ]:
SBP_snps_1 = SBP_snps_5[SBP_snps_5['phenotype'] == 'SBP_pre_1_ADJ']

In [ ]:
SBP_snps_2 = SBP_snps_5[SBP_snps_5['phenotype'] == 'SBP_pre_2_ADJ']

In [ ]:
SBP_snps_3 = SBP_snps_5[SBP_snps_5['phenotype'] == 'SBP_pre_3_ADJ']

In [ ]:
SBP_snps_4 = SBP_snps_5[SBP_snps_5['phenotype'] == 'SBP_pre_4_ADJ']

In [ ]:
DBP_snps_5 = pd.read_csv(
    'prepost_dbp/cusk/e3_l3_d1/merged_blocks_selected_markers.tsv',
    sep = '\t'
)

In [ ]:
DBP_snps_1 = DBP_snps_5[DBP_snps_5['phenotype'] == 'DBP_pre_1_ADJ']

In [ ]:
DBP_snps_2 = DBP_snps_5[DBP_snps_5['phenotype'] == 'DBP_pre_2_ADJ']

In [ ]:
DBP_snps_3 = DBP_snps_5[DBP_snps_5['phenotype'] == 'DBP_pre_3_ADJ']

In [ ]:
DBP_snps_4 = DBP_snps_5[DBP_snps_5['phenotype'] == 'DBP_pre_4_ADJ']

In [ ]:
bin_only = pd.read_csv(
    'twoyears_bin_only/cusk/e3_l3_d1/merged_blocks_selected_markers.tsv',
    sep='\t')

bin_only_on_drugs = pd.read_csv(
    'twoyears_on_drugs_bin_only/cusk/e3_l3_d1/merged_blocks_selected_markers.tsv',
    sep='\t'
)

In [ ]:
bin_only

In [ ]:
bin_only_on_drugs

### single-snp associations tests:

In [ ]:
rsids = list(set(list(DBP_snps_all['rsID']) + 
                 list(SBP_snps_all['rsID']) + 
                 list(SBP_snps_1['rsID']) +
                 list(DBP_snps_1['rsID']) +
                 list(SBP_snps_2['rsID']) +
                 list(DBP_snps_2['rsID']) +
                 list(SBP_snps_3['rsID']) +
                 list(DBP_snps_3['rsID']) +
                 list(SBP_snps_4['rsID']) +
                 list(DBP_snps_4['rsID'])
                )
            )

In [ ]:
#now save the list of rsids and eid to filter the bed file to load it easily:
#pd.DataFrame(rsids).to_csv('snps_to_keep.txt', index=False, header=False)

In [ ]:
bed = bed_reader.open_bed('filtered_for_arb_response.bed') #this is only filtered for participants
val = bed.read()

In [ ]:
my_snps = pd.DataFrame(val, columns = bed.sid)

In [ ]:
my_snps['eid'] = (bed.iid).astype(int)

In [ ]:
my_snps = my_snps[my_snps['eid'].isin(aggregated.eid)]

In [ ]:
aggregated = pd.read_csv('agg_arb.csv')

In [ ]:
aggregated = aggregated.merge(my_snps, on='eid')

In [ ]:
aggregated.to_csv('agg_arb_with_bp_snps.csv')

### Multiple regression

In [ ]:
aggregated = pd.read_csv('agg_arb_with_bp_snps.csv')

In [ ]:
def load_covariates(cov_file, cov_names_file, exclude_keywords=("centre","batch")):

    cov_names = []
    with open(cov_names_file, 'r') as fin:
        for line in fin:
            cov_names.append(line.strip())  

    all_rows = []
    with open(cov_file, 'r') as fin:
        for line in fin:
            fields = line.split()
            fields_floats = [float(x) for x in fields]
            all_rows.append(fields_floats)

    cov_df = pd.DataFrame(all_rows, columns=cov_names)

    cov_df["IID"] = cov_df["IID"].astype(int)
    cov_df["FID"] = cov_df["FID"].astype(int)

    exclude_cols = []
    for col in cov_df.columns:
        col_lower = col.lower()
        if any(kw in col_lower for kw in exclude_keywords):
            exclude_cols.append(col)
     
    cov_df.rename(columns={"IID":"eid"}, inplace=True)

    cov_df.drop(columns=exclude_cols, inplace=True, errors="ignore")

    return cov_df

In [ ]:
cov_file = "UKB_covars_noAge.cov"
cov_names_file = "UKB_covars_noAge.Namecov"
df_cov = load_covariates(cov_file, cov_names_file, exclude_keywords=("centre","batch"))

In [ ]:
aggregated = pd.merge(aggregated, df_cov, on="eid", how="left")

### load in BP data as well

In [ ]:
bp_5 = pd.read_csv(
    'pre_post_cont_traits.tsv',
    sep = '\t'
)

In [ ]:
aggregated = pd.merge(aggregated, bp_5[['eid','SBP_pre_1', 'SBP_pre_2', 'SBP_pre_3',
       'SBP_pre_4', 'SBP_pre_5']], on="eid", how="left")

In [ ]:
bp_all = pd.read_csv(
    'single_age_pre_post_twoyears_on_drugs.tsv',
    sep = '\t')

In [ ]:
aggregated = pd.merge(aggregated, bp_all[['eid','SBP_pre']], on="eid", how="left")

In [ ]:
def adjust_and_standardize_phenotypes(df, phenotype_cols, covariate_cols):

    out_df = df.copy()
    
    for pheno in phenotype_cols:
        needed_cols = covariate_cols + [pheno]
        
        sub = out_df[needed_cols].dropna(subset=[pheno])
        if len(sub) < 2 or sub[pheno].nunique() < 2:
            print("no data or variance")

        X = sub[covariate_cols].values
        y = sub[pheno].values

        imp = SimpleImputer(missing_values=np.nan, strategy='mean')
        X_imp = imp.fit_transform(X)

        lm = LinearRegression()
        lm.fit(X_imp, y)
        y_pred = lm.predict(X_imp)
        residuals = y - y_pred

        out_df[pheno + "_AD"] = np.nan
        out_df.loc[sub.index, pheno + "_AD"] = residuals

        valid_mask = ~pd.isna(out_df[pheno + "_AD"])
        if valid_mask.sum() > 1:
            rvals = out_df.loc[valid_mask, pheno + "_AD"].values
            mean_r = np.mean(rvals)
            std_r = np.std(rvals, ddof=1)  # sample std
            if std_r > 0:
                zvals = (rvals - mean_r) / std_r
            else:
                zvals = rvals * 0.0
            out_df[pheno + "_ADJ"] = np.nan
            out_df.loc[valid_mask, pheno + "_ADJ"] = zvals
        else:
            out_df[pheno + "_ADJ"] = np.nan

    return out_df        

In [ ]:
covariate_cols = [c for c in df_cov.columns if c not in ("eid","FID")]

In [ ]:
phenotype_cols = [
        'num_arb_therapies',
        'num_diff_drug_classes',
        'num_diff_arbs',
        'longest_arb_duration',
        'avg_dose_Candesartan Cilexetil',
        'avg_dose_Eprosartan',
        'avg_dose_Irbesartan',
        'avg_dose_Losartan Potassium',
        'avg_dose_Olmesartan',
        'avg_dose_Telmisartan',
        'avg_dose_Valsartan',
        'dose_ever_increased',
        'arb_is_longest_therapy',
        'arb_is_last_therapy',
        'arb_is_first_therapy',
        'arb_ever_augmented',
        'changed_from_arb',
        'SBP_pre_1',
        'SBP_pre_2',
        'SBP_pre_3',
        'SBP_pre_4',
        'SBP_pre_5',
        'SBP_pre'
]

In [ ]:
df = adjust_and_standardize_phenotypes(aggregated, phenotype_cols, covariate_cols)

### calculate and visualise correlations between the different phenotypes

In [ ]:
df = pd.read_csv('agg_arb_bwith_bp_snps_and_bp.csv',
              sep = '\t')

In [ ]:
adjz_cols = [c for c in df.columns if c.endswith("_ADJ")]
corr_matrix = df[adjz_cols].corr(method="pearson")
plt.figure(figsize=(17, 17))
sns.heatmap(corr_matrix, annot=True, cmap="vlag", square=True)
plt.tight_layout()
plt.savefig('adjusted_standardised_pearson_heatmap.png')

In [ ]:
df.to_csv('<OUTPUT_DIR>/agg_arb_bwith_bp_snps_and_bp.csv',
              sep = '\t')

In [ ]:
def adjust_r_squared(r2, N, k):
    if N <= k + 1:
        return np.nan  
    return r2 - (k * (1 - r2) / (N - k - 1)) #N samples k snps

In [ ]:
def run_regressions(
    df,
    phenotype_cols,
    snp_sets,
    sbp_col="SBP_pre_ADJ",
    dropna_phenotype=True,
    outdir="./regression_results"
):
    if not os.path.exists(outdir):
        os.makedirs(outdir)

    def fit_and_save_regression(X_cols, y_col, outname):
        sub_cols = X_cols + [y_col]
        sub_df = df[sub_cols].copy()
        if dropna_phenotype:
            sub_df.dropna(subset=[y_col], inplace=True)
        if len(sub_df) < 2 or sub_df[y_col].nunique() < 2:
            return

        X = sub_df[X_cols].values
        y = sub_df[y_col].values

        imp = SimpleImputer(strategy='mean')
        X_imp = imp.fit_transform(X)

        lm = LinearRegression()
        lm.fit(X_imp, y)

        r2 = lm.score(X_imp, y)
        N = X_imp.shape[0]
        k = X_imp.shape[1]
        r2_adj = adjust_r_squared(r2, N, k)

        rows = []
        rows.append({
            "predictor": "Intercept",
            "coefficient": lm.intercept_,
            "n_samples": N,
            "n_predictors": k,
            "r2": r2,
            "adjusted_r2": r2_adj
        })
        for col, coef_val in zip(X_cols, lm.coef_):
            rows.append({
                "predictor": col,
                "coefficient": coef_val,
                "n_samples": N,
                "n_predictors": k,
                "r2": r2,
                "adjusted_r2": r2_adj
            })

        pd.DataFrame(rows).to_csv(outname, index=False)

    for pheno in phenotype_cols:

        if sbp_col in df.columns:
            outname_sbp = os.path.join(outdir, f"reg_SBPonly_{pheno}.csv")
            fit_and_save_regression([sbp_col], pheno, outname_sbp)

        for model_name, snp_list in snp_sets.items():
            snp_cols = [s for s in snp_list if s in df.columns]
            if not snp_cols:
                print(f"No valid SNP columns for {model_name}, skipping.")
                continue

            
            outname_snps = os.path.join(
                outdir, f"reg_{model_name}_{pheno}_snpsOnly.csv"
            )
            fit_and_save_regression(snp_cols, pheno, outname_snps)

            
            if sbp_col in df.columns:
                outname_snps_sbp = os.path.join(
                    outdir, f"reg_{model_name}_{pheno}_snpsPlusSBP.csv"
                )
                fit_and_save_regression(snp_cols + [sbp_col], pheno, outname_snps_sbp)



In [ ]:
snp_sets = {
       "DBP_snps_all": list(DBP_snps_all["rsID"]),
       "SBP_snps_all": list(SBP_snps_all["rsID"]),
       "SBP_snps_1":   list(SBP_snps_1["rsID"]),
       "DBP_snps_1":   list(DBP_snps_1["rsID"]),
       "SBP_snps_2":   list(SBP_snps_2["rsID"]),
       "DBP_snps_2":   list(DBP_snps_2["rsID"]),
       "SBP_snps_3":   list(SBP_snps_3["rsID"]),
       "DBP_snps_3":   list(DBP_snps_3["rsID"]),
       "SBP_snps_4":   list(SBP_snps_4["rsID"]),
       "DBP_snps_4":   list(DBP_snps_4["rsID"]),
       "all_snps": rsids
    }

phenotype_cols = [
        'num_arb_therapies_ADJ',
        'num_diff_drug_classes_ADJ',
        'num_diff_arbs_ADJ',
        'longest_arb_duration_ADJ',
        'avg_dose_Candesartan Cilexetil_ADJ',
        'avg_dose_Eprosartan_ADJ',
        'avg_dose_Irbesartan_ADJ',
        'avg_dose_Losartan Potassium_ADJ',
        'avg_dose_Olmesartan_ADJ',
        'avg_dose_Telmisartan_ADJ',
        'avg_dose_Valsartan_ADJ',
        'dose_ever_increased_ADJ',
        'arb_is_longest_therapy_ADJ',
        'arb_is_last_therapy_ADJ',
        'arb_is_first_therapy_ADJ',
        'arb_ever_augmented_ADJ',
        'changed_from_arb_ADJ'
]

In [ ]:
run_regressions(
    df,
    phenotype_cols,
    snp_sets,
    sbp_col="SBP_pre_ADJ",
    dropna_phenotype=True,
    outdir="./regression_results"
)

In [ ]:
def read_regs(directory="regression_results/adjusted/"):

    files = glob.glob(os.path.join(directory, "reg_*.csv"))
    all_dfs = []
    for file_path in files:
        fname = os.path.basename(file_path)
        model_name, phenotype, model_type = parse_filename_for_models(fname)
        if model_name is None:
            print(f"Skipping '{fname}' - parse failed.")
            continue

        df_csv = pd.read_csv(file_path)
        df_csv["model_name"] = model_name
        df_csv["phenotype"] = phenotype
        df_csv["model_type"] = model_type
        all_dfs.append(df_csv)

    if not all_dfs:
        print(f"No matching files found in {directory}")
        return pd.DataFrame()

    big_df = pd.concat(all_dfs, ignore_index=True)
    return big_df

In [ ]:
def parse_filename_for_models(filename):

    core = filename
    if core.lower().startswith("reg_"):
        core = core[4:]
    if core.lower().endswith(".csv"):
        core = core[:-4]

    tokens = core.split("_")
    if not tokens:
        return None, None, None

    last_tok = tokens[-1].lower()
    if last_tok in ("snpsonly","snpsplussbp"):
        model_type = last_tok
        tokens = tokens[:-1] 
    else:
        model_type = ""

    model_name = None
    phenotype_tokens = []

    def join_tokens_for_model(toks):
        return "_".join(toks)

    if tokens[0].lower() == "sbponly":
        model_name = "SBPonly"
        phenotype_tokens = tokens[1:] 
        
    elif len(tokens) >= 2 and tokens[0].lower()=="all" and tokens[1].lower()=="snps":
        model_name = "all_snps"
        phenotype_tokens = tokens[2:]

    elif len(tokens) >= 3 and tokens[1].lower()=="snps":

        model_name = f"{tokens[0]}_snps_{tokens[2]}"
        phenotype_tokens = tokens[3:]
    else:
        return None, None, None

    phenotype = "_".join(phenotype_tokens)
    return model_name, phenotype, model_type

In [ ]:
def summarise_regs(big_df):
    summary_rows = []
    group_cols = ["model_name", "phenotype", "model_type"]
    grouped = big_df.groupby(group_cols, as_index=False)

    for (m_name, pheno, m_type), gdf in grouped:
        if gdf.empty:
            continue
        first_row = gdf.iloc[0]
        r2_val = first_row.get("r2", float("nan"))
        r2_adj_val = first_row.get("adjusted_r2", float("nan"))

        df_preds = gdf[gdf["predictor"] != "Intercept"]
        n_preds = len(df_preds)

        if not df_preds.empty:
            idx_top = df_preds["coefficient"].abs().idxmax()
            row_top = df_preds.loc[idx_top]
            top_pred = row_top["predictor"]
            top_coef = row_top["coefficient"]
        else:
            top_pred = float("nan")
            top_coef = float("nan")

        summary_rows.append({
            "model_name": m_name,
            "phenotype": pheno,
            "model_type": m_type,
            "top_predictor": top_pred,
            "top_predictor_coef": top_coef,
            "n_predictors": n_preds,
            "r2": r2_val,
            "adjusted_r2": r2_adj_val
        })

    result_df = pd.DataFrame(summary_rows)
    result_df.sort_values(group_cols, inplace=True)
    result_df.reset_index(drop=True, inplace=True)
    return result_df

In [ ]:
regs = read_regs()

In [ ]:
regs

In [ ]:
summary = summarize_regs(regs)

In [ ]:
summary

In [ ]:
summary.to_csv('regs_with_sbp.tsv', sep = '\t')